# CREDIT CARD FRAUD DETECTION

# Import Libraries

In [ ]:
# Data Handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, roc_curve

# Imbalance Handling
from imblearn.over_sampling import SMOTE

# Save model
import joblib

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")


In [ ]:
df = pd.read_csv("/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv")
df.head()


In [ ]:
df.info()
df.describe()


# Exploratory Data Analysis

# Class Distribution

In [ ]:
sns.countplot(x='Class', data=df)
plt.title("Fraud vs Genuine Transactions")
plt.show()

print(df['Class'].value_counts())


# 
Percentage of Fraud

In [ ]:
fraud_percent = (df['Class'].value_counts()[1] / len(df)) * 100
print("Fraud Percentage:", fraud_percent)


# Correlation Heatmap

In [ ]:
plt.figure(figsize=(20,15))
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()



# Data Preprocessing

# Normalize Time and Amount

In [ ]:
scaler = StandardScaler()

df['scaled_amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_time'] = scaler.fit_transform(df[['Time']])

df.drop(['Time','Amount'], axis=1, inplace=True)


# Split Data

In [ ]:

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# Handle Class Imbalance

In [ ]:
smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_resampled.value_counts())


# Model 1 – Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_resampled, y_train_resampled)

y_pred_lr = lr.predict(X_test)

print("Logistic Regression Results")
print(classification_report(y_test, y_pred_lr))


# Model 2 – Random Forest

In [ ]:

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,              # Use all CPU cores
    class_weight='balanced' # Handle imbalance automatically
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Results")
print(classification_report(y_test, y_pred_rf))


# Model 3 – XGBoost

In [ ]:
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1)
xgb.fit(X_train_resampled, y_train_resampled)

y_pred_xgb = xgb.predict(X_test)

print("XGBoost Results")
print(classification_report(y_test, y_pred_xgb))


# Model Evaluation Comparison

In [ ]:
models = {
    "Logistic Regression": lr,
    "Random Forest": rf,
    "XGBoost": xgb
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    print("Model:", name)
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("ROC AUC:", roc_auc_score(y_test, y_pred))
    print("-"*50)



# Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

grid = GridSearchCV(
    rf,
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1,   # Parallel processing
    verbose=2
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)


# Save Best Model



In [ ]:


best_model = grid.best_estimator_


In [ ]:
best_model = grid.best_estimator_
print("Best Parameters:", grid.best_params_)


In [ ]:
import os
import joblib

# Create models directory
base_path = os.getcwd()
model_dir = os.path.join(base_path, "models")
os.makedirs(model_dir, exist_ok=True)

# Save model
model_path = os.path.join(model_dir, "best_model.pkl")
joblib.dump(best_model, model_path)

print("Model saved at:", model_path)


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

y_probs = best_model.predict_proba(X_test)[:,1]

fpr, tpr, thresholds = roc_curve(y_test, y_probs)
auc_score = roc_auc_score(y_test, y_probs)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.4f}")
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

print("ROC-AUC Score:", auc_score)


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, thresholds = precision_recall_curve(y_test, y_probs)
ap_score = average_precision_score(y_test, y_probs)

plt.figure(figsize=(6,6))
plt.plot(recall, precision, label=f"AP = {ap_score:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()

print("Average Precision Score:", ap_score)


In [ ]:
# Prediction Function (Production Ready)
def predict_transaction(input_data):
    model = joblib.load("models/best_model.pkl")
    prediction = model.predict([input_data])
    return "Fraud" if prediction[0] == 1 else "Genuine"

In [ ]:
def predict_transaction(input_data, threshold=0.5):
    """
    Predict if a transaction is Fraud or Genuine
    
    Parameters:
    input_data : list or array (single transaction features)
    threshold : decision threshold (default=0.5)
    
    Returns:
    dict with prediction and probability
    """
    
    # Convert to numpy array
    input_array = np.array(input_data).reshape(1, -1)
    
    # Get probability
    prob = model.predict_proba(input_array)[0][1]
    
    # Apply custom threshold
    prediction = 1 if prob >= threshold else 0
    
    return {
        "Fraud Probability": round(float(prob), 4),
        "Prediction": "Fraud" if prediction == 1 else "Genuine"
    }


In [ ]:
sample_transaction = X_test.iloc[0].values

result = predict_transaction(sample_transaction, threshold=0.3)

print(result)
